In [ ]:
import pandas as pd
import numpy as np

# Load and prepare data

In [ ]:
fields = pd.read_csv("outputs/2.0-discovery_results.csv")
canonical = pd.read_csv(
    "outputs/4.2-final_labeled_field_profiles.csv",
    usecols=["metadata_field", "canonical_name", "confidence"],
)

In [ ]:
df = fields.merge(canonical, on="metadata_field", how="left")
df["confidence"] = df["confidence"].map({"high": 3, "medium": 2, "low": 1})
df

In [ ]:
df[df["canonical_name"].isna()]

# Frequency analysis

## Canonical field support

In [ ]:
snapshot_count = df["snapshot_id"].nunique()
df["figure_snapshot_id"] = np.where(
    df["snapshot_type"] == "figure", df["snapshot_id"], None
)
df["table_snapshot_id"] = np.where(
    df["snapshot_type"] == "table", df["snapshot_id"], None
)

agg1 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    snapshot_support=("snapshot_id", lambda x: x.nunique() / snapshot_count),
    corpora_count=("source", "nunique"),
    figure_count=("figure_snapshot_id", "nunique"),
    table_count=("table_snapshot_id", "nunique"),
    avg_confidence=("confidence", "mean"),
)

agg1

## By corpus

In [ ]:
df["unhcr_snapshot_id"] = np.where(df["source"] == "unhcr", df["snapshot_id"], None)
df["prwp_snapshot_id"] = np.where(df["source"] == "prwp", df["snapshot_id"], None)
df["refugee_snapshot_id"] = np.where(df["source"] == "refugee", df["snapshot_id"], None)

agg2 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    unhcr_count=("unhcr_snapshot_id", "nunique"),
    prwp_count=("prwp_snapshot_id", "nunique"),
    refugee_count=("refugee_snapshot_id", "nunique"),
)

agg2

## By snapshot type

In [ ]:
agg3 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    figure_count=("figure_snapshot_id", "nunique"),
    table_count=("table_snapshot_id", "nunique"),
)

agg3

## By source level

In [ ]:
df["snapshot_snapshot_id"] = np.where(
    df["source_level"] == "snapshot", df["snapshot_id"], None
)
df["document_snapshot_id"] = np.where(
    df["source_level"] == "document", df["snapshot_id"], None
)
df["both_snapshot_id"] = np.where(df["source_level"] == "both", df["snapshot_id"], None)


agg4 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    source_snapshot_count=("snapshot_snapshot_id", "nunique"),
    source_document_count=("document_snapshot_id", "nunique"),
    source_both_count=("both_snapshot_id", "nunique"),
)

agg4

## Diversity

In [ ]:
agg5 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    n_unique_observed_values=("observed_value", "nunique"),
)

agg5

## Alias richness

In [ ]:
agg6 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    snapshot_support=("snapshot_id", lambda x: x.nunique() / snapshot_count),
    alias_count=("metadata_field", "nunique"),
)

agg6

## Support scoring

In [ ]:
# support_score = (snapshot_support) x (corpora_count / 3) × (avg_confidence / 3)

agg1["support_score"] = (
    agg1["snapshot_support"] * agg1["corpora_count"] / 3 * agg1["avg_confidence"] / 3
)

agg1.sort_values("support_score", ascending=False)

## Alias count vs Snapshot support 

In [ ]:
import plotly.express as px

In [ ]:
px.scatter(
    agg6.reset_index(),
    x="snapshot_support",
    y="alias_count",
    hover_name="canonical_name",
    title="Alias count vs Snapshot support",
    width=800,
    height=500,
)

## Ontology compression

In [ ]:
agg7 = df.groupby("canonical_name").agg(
    snapshot_count=("snapshot_id", "nunique"),
    profile_count=("canonical_name", "count"),
)
agg7["compression"] = (agg7["profile_count"] - agg7["snapshot_count"]) / agg7["snapshot_count"]

agg7.sort_values("compression", ascending=False)

In [ ]:
# from pandas.io.clipboard import clipboard_set

# clipboard_set(agg6.to_markdown())